As regras de negocios seguiram o mesmo padrao das queries feitas anteriormente no notebook da camada gold. Tambem criei esse notebook para não precisar fazer um novo commit depois da entrega do notebook da camada gold da atividade 2.

In [0]:
import pyspark.sql.functions as F

catalog = "workspace"
schema_ouro = f"{catalog}.gold"

objetos_necessarios = [
    "dim_people", "dim_movies", "dim_generos","dim_producers", "dim_reviews",
    "bridge_movie_company", "fact_movies_performance", "bridge_movie_genre","bridge_movie_person"
]

for objeto in objetos_necessarios:
    nome_completo = f"{schema_ouro}.{objeto}"
    assert spark.catalog.tableExists(nome_completo), f"Execute o notebook 03: falta {nome_completo}"

print("✅ Contrato da Gold disponível")

In [0]:
spark.sql(f"""CREATE OR REPLACE VIEW {catalog}.gold.vw_receita_total AS
          SELECT
          SUM(receita_brl) AS receita_total_em_real
          FROM {catalog}.gold.fact_movies_performance""")

display(spark.table(f"{catalog}.gold.vw_receita_total"))

In [0]:
spark.sql(f"""CREATE OR REPLACE VIEW {catalog}.gold.vw_top5_populares AS
          SELECT d.titulo, f.popularidade
          FROM {catalog}.gold.fact_movies_performance f
          JOIN {catalog}.gold.dim_movies d
          ON f.sk_movie_id = d.sk_movie_id
          ORDER BY f.popularidade DESC
          LIMIT 5""")
display(spark.table(f"{catalog}.gold.vw_top5_populares"))



In [0]:
spark.sql(f"""CREATE OR REPLACE VIEW {catalog}.gold.vw_generos AS
          SELECT g.nome_genero, COUNT(b.sk_movie_id) AS qtd_filmes
          FROM {catalog}.gold.bridge_movie_genre b
          INNER JOIN {catalog}.gold.dim_generos g ON b.sk_genre_id = g.sk_genre_id
          GROUP BY g.nome_genero
          ORDER BY qtd_filmes DESC
          """)
display(spark.table(f"{catalog}.gold.vw_generos"))


In [0]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.vw_top_10_receita_rank AS
    SELECT 
        d.titulo, 
        f.receita_usd, 
        f.receita_brl,
        RANK() OVER (ORDER BY f.receita_brl DESC) AS posicao_rank
    FROM {catalog}.gold.fact_movies_performance f
    INNER JOIN {catalog}.gold.dim_movies d ON f.sk_movie_id = d.sk_movie_id
    ORDER BY posicao_rank
    LIMIT 10
""")
display(spark.table(f"{catalog}.gold.vw_top_10_receita_rank"))

In [0]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.vw_ator_mais_participacoes_2_anos AS
    WITH DataLimite AS (
        SELECT MAX(data_lancamento) AS max_data
        FROM {catalog}.gold.dim_movies
        WHERE data_lancamento <= CURRENT_DATE()
    )
    SELECT 
        p.nome_pessoa, 
        COUNT(b.sk_movie_id) AS qtd_participacoes
    FROM {catalog}.gold.bridge_movie_person b
    INNER JOIN {catalog}.gold.dim_people p ON b.sk_person_id = p.sk_person_id
    INNER JOIN {catalog}.gold.dim_movies d ON b.sk_movie_id = d.sk_movie_id
    CROSS JOIN DataLimite m
    WHERE p.tipo_pessoa = 'Ator'
      AND d.data_lancamento BETWEEN (m.max_data - INTERVAL 2 YEARS) AND m.max_data
    GROUP BY p.nome_pessoa
    ORDER BY qtd_participacoes DESC
    LIMIT 1
""")
display(spark.table(f"{catalog}.gold.vw_ator_mais_participacoes_2_anos"))

In [0]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.vw_produtora_maior_lucro_5_anos AS
    WITH DataLimite AS (
        SELECT MAX(data_lancamento) AS max_data
        FROM {catalog}.gold.dim_movies
        WHERE data_lancamento <= CURRENT_DATE()
    )
    SELECT 
        p.nome_produtora, 
        SUM(f.lucro_usd) AS lucro_total_usd, 
        SUM(f.lucro_brl) AS lucro_total_brl
    FROM {catalog}.gold.bridge_movie_company b
    INNER JOIN {catalog}.gold.dim_producers p ON b.sk_company_id = p.sk_company_id
    INNER JOIN {catalog}.gold.fact_movies_performance f ON b.sk_movie_id = f.sk_movie_id
    INNER JOIN {catalog}.gold.dim_movies d ON b.sk_movie_id = d.sk_movie_id
    CROSS JOIN DataLimite m
    WHERE d.data_lancamento BETWEEN (m.max_data - INTERVAL 5 YEARS) AND m.max_data
    GROUP BY p.nome_produtora
    ORDER BY lucro_total_usd DESC
    LIMIT 1
""")
display(spark.table(f"{catalog}.gold.vw_produtora_maior_lucro_5_anos"))